In [18]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

DROP_COLUMNS = ["Contact Name", "Account Name"]

def read_salesforce_report(path):
    raw = pd.read_excel(path, header=None, dtype=str)
    header_row = raw.index[(raw == "Case Number").any(axis=1)][0]

    df = raw.iloc[header_row + 1:].copy()
    df.columns = raw.iloc[header_row].str.strip()
    df = df.loc[:, df.columns.notna()]
    df = df.dropna(axis=1, how="all")

    owner_col = [c for c in df.columns if str(c).startswith("Case Owner")][0]
    df = df.rename(columns={owner_col: "Case Owner"})

    df = df[~df["Case Owner"].isin(["Subtotal", "Total"])]
    df["Case Owner"] = df["Case Owner"].ffill()
    df = df[df["Case Number"].astype(str).str.isdigit()]

    df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])
    return df.reset_index(drop=True)

open_df = read_salesforce_report("open_cases.xlsx")
closed_df = read_salesforce_report("closed_cases.xlsx")

print("Open cases:  ", len(open_df), "rows")
print("Closed cases:", len(closed_df), "rows")

Open cases:   162 rows
Closed cases: 19771 rows


In [2]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable


In [19]:
open_df = open_df.rename(columns={
    "Case Owner": "case_owner",
    "Case Number": "case_number",
    "Sub-Status": "sub_status",
    "Case Record Type": "record_type",
    "Source": "source",
    "Date/Time Opened": "opened_at",
    "Next Follow Up Date/Time": "next_follow_up",
    "Case Date/Time Last Modified": "last_modified",
})

closed_df = closed_df.rename(columns={
    "Case Owner": "case_owner",
    "Case Number": "case_number",
    "Source": "source",
    "Closed Date": "closed_at",
    "Opened Date": "opened_at",
    "Case Last Modified Date": "last_modified",
    "Product Type": "product_type",
    "Client is Onsite": "client_onsite",
    "Sub-Status": "sub_status",
})

print(list(open_df.columns))
print(list(closed_df.columns))

['case_owner', 'case_number', 'sub_status', 'record_type', 'source', 'opened_at', 'next_follow_up', 'last_modified']
['case_owner', 'case_number', 'source', 'closed_at', 'opened_at', 'last_modified', 'product_type', 'client_onsite', 'sub_status']


In [20]:
for col in ["opened_at", "next_follow_up", "last_modified"]:
    open_df[col] = pd.to_datetime(open_df[col], format="%m/%d/%Y %I:%M %p", errors="coerce")

for col in ["opened_at", "closed_at", "last_modified"]:
    closed_df[col] = pd.to_datetime(closed_df[col], format="%m/%d/%Y", errors="coerce")

print(open_df.dtypes)

13
case_owner                object
case_number               object
sub_status                object
record_type               object
source                    object
opened_at         datetime64[ns]
next_follow_up    datetime64[ns]
last_modified     datetime64[ns]
dtype: object


In [21]:
for df in (open_df, closed_df):
    df["case_owner"] = df["case_owner"].str.strip().str.title()

for col in ["source", "product_type", "client_onsite", "sub_status"]:
    closed_df[col] = closed_df[col].fillna("Unknown").replace("", "Unknown")
open_df["source"] = open_df["source"].fillna("Unknown")

print(closed_df["source"].value_counts())

CSC                  6393
Community            4025
Sales                3945
Web to Case          1683
Inbound Call         1378
Unknown              1282
Omni Channel          507
Internal              314
Email                 144
eCommerce Emails       53
planner-assistant      44
Sales/Internal          2
Mobile Release          1
Name: source, dtype: int64


In [22]:
common_reps = set(open_df["case_owner"]) & set(closed_df["case_owner"])
removed = set(open_df["case_owner"]) - common_reps
print("Removed from open data:", removed)

open_df = open_df[open_df["case_owner"].isin(common_reps)]
print("Open cases now:", len(open_df))

Removed from open data: {'Avni Sharma'}
Open cases now: 143


In [23]:
USE_REP_CODES = True

if USE_REP_CODES:
    all_reps = sorted(set(open_df["case_owner"]) | set(closed_df["case_owner"]))
    rep_codes = {name: f"Rep {i:02d}" for i, name in enumerate(all_reps, start=1)}
    open_df["case_owner"] = open_df["case_owner"].map(rep_codes)
    closed_df["case_owner"] = closed_df["case_owner"].map(rep_codes)
    pd.Series(rep_codes).to_csv("rep_code_key_PRIVATE.csv", header=["rep_code"])

print(sorted(closed_df["case_owner"].unique()))

['Rep 01', 'Rep 02', 'Rep 03', 'Rep 04', 'Rep 05', 'Rep 06', 'Rep 07', 'Rep 08', 'Rep 09', 'Rep 10', 'Rep 11', 'Rep 12', 'Rep 13']


In [24]:
closed_df["days_to_close"] = (closed_df["closed_at"] - closed_df["opened_at"]).dt.days

report_time = open_df["last_modified"].max()
open_df["days_open"] = (report_time - open_df["opened_at"]).dt.days
open_df["follow_up_overdue"] = (open_df["next_follow_up"] < report_time).astype(int)

closed_df[["case_number", "opened_at", "closed_at", "days_to_close"]].head()

14,case_number,opened_at,closed_at,days_to_close
0,30294151,2024-10-24,2024-12-16,53
1,30438548,2024-11-11,2024-12-05,24
2,30505054,2024-11-20,2024-12-05,15
3,30512713,2024-11-21,2024-12-05,14
4,30519605,2024-11-22,2024-12-13,21


In [25]:
open_part = open_df.assign(state="Open", closed_at=pd.NaT)
closed_part = closed_df.assign(state="Closed")

common_cols = ["case_number", "case_owner", "source", "opened_at", "closed_at", "state"]
all_cases = pd.concat([open_part[common_cols], closed_part[common_cols]], ignore_index=True)

print("Total rows:", len(all_cases))
print("Duplicate case numbers:", all_cases["case_number"].duplicated().sum())
print(all_cases["state"].value_counts())

Total rows: 19914
Duplicate case numbers: 0
Closed    19771
Open        143
Name: state, dtype: int64


In [26]:
print("Unknown source:", (closed_df["source"] == "Unknown").sum(),
      f"({(closed_df['source'] == 'Unknown').mean():.1%})")
print("Negative days_to_close:", (closed_df["days_to_close"] < 0).sum())
print("Open cases missing follow-up date:", open_df["next_follow_up"].isna().sum())
print("Median days to close:", closed_df["days_to_close"].median())
print("Closed date range:", closed_df["closed_at"].min().date(), "to", closed_df["closed_at"].max().date())

Unknown source: 1282 (6.5%)
Negative days_to_close: 0
Open cases missing follow-up date: 8
Median days to close: 6.0
Closed date range: 2024-12-01 to 2026-10-13


In [27]:
open_df.to_csv("open_cases_clean.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
closed_df.to_csv("closed_cases_clean.csv", index=False, date_format="%Y-%m-%d")
all_cases.to_csv("all_cases_clean.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")

print("Saved 3 files")

Saved 3 files
